# Exercise Sheet 3 — Least squares: worked solutions
DM587

Run the cells from top to bottom in a fresh Python kernel. Keep `housing.csv` beside the notebook. Packages: NumPy, SciPy and Matplotlib. The notebook creates `housing.npy` in the current folder in Question 6. All other data are included in the cells.

These are standard notebook cells with ordinary Matplotlib outputs, suitable for Jupyter and PyCharm with notebook support. The clean notebook contains the same explanations and code with outputs removed. Timing and ill-conditioned numerical results vary by machine; the outputs here are one recorded run.

Each solution is labelled with its question and subquestion. Mathematical proofs use Markdown; numerical parts show the computation and its output. No `assert` statements are needed.

In [ ]:
import numpy as np
import scipy
from scipy.linalg import solve_triangular
import matplotlib.pyplot as plt
import timeit
from pathlib import Path

np.set_printoptions(precision=6, suppress=False)
plt.rcParams.update({"figure.figsize": (8, 4), "font.size": 11})
print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)


## Question 1 — Shorthand operators
For NumPy arrays:

| Operator | Function form | Meaning |
| --- | --- | --- |
| `A @ B` | `np.matmul(A, B)` | Matrix multiplication |
| `A * B` | `np.multiply(A, B)` | Elementwise multiplication |

With one-dimensional vectors, `@` gives a dot product; a matrix and a vector give a matrix-vector product. Matching inner dimensions must agree. Use `*` for scalar multiplication too. Elementwise products can broadcast compatible unequal shapes.

### (a) Linear-algebra products
`@` and `np.matmul` express the same product for NumPy arrays. For the vectors and matrices here, `np.dot` agrees as well. Two length-3 vectors produce a scalar dot product; a $(2,3)$ matrix times a length-3 vector produces a length-2 vector; a $(2,3)$ matrix times a $(3,2)$ matrix produces a $(2,2)$ matrix. A scalar's shape is `()`, while a one-dimensional length-2 vector has shape `(2,)`.

In [ ]:
v = np.array([1., 2., 3.])
w = np.array([4., 5., 6.])
A_at = np.array([[1., 2., 3.],
                 [4., 5., 6.]])
B_at = np.array([[1., 0.],
                 [0., 1.],
                 [1., 1.]])
C_at = np.array([[2., 1., 0.],
                 [1., 0., 2.]])
matrix_vector = A_at @ v
matrix_matrix = A_at @ B_at
for label, left, right in [("v @ w", v, w),
                            ("A @ v", A_at, v),
                            ("A @ B", A_at, B_at)]:
    result = left @ right
    print(label, "=", result)
    print("Shape:", np.shape(result))
    print("Agrees with matmul:", np.allclose(result, np.matmul(left, right)))
    print("Agrees with dot:", np.allclose(result, np.dot(left, right)))


### (b) Elementwise products and broadcasting
`v * w` produces three individual products rather than their sum. `A * C` produces a $(2,3)$ matrix because both inputs have that shape. `np.multiply` gives the same results.

`A * v` also produces a $(2,3)$ array: the three vector entries scale the three columns in each row. In contrast, `A @ v` sums each row's products to give a length-2 vector. Scalar multiplication scales every entry of the vector or matrix.

In [ ]:
print("v * w:", v * w)
print("Agrees with multiply:", np.allclose(v * w, np.multiply(v, w)))
print("A * C:")
print(A_at * C_at)
print("Agrees with multiply:", np.allclose(A_at * C_at, np.multiply(A_at, C_at)))
print("A * v:")
print(A_at * v)
print("Shape:", (A_at * v).shape)
print("A @ v:", A_at @ v)
print("Shape:", (A_at @ v).shape)
print("2 * v:", 2 * v)
print("2 * A:")
print(2 * A_at)


### (c) Invalid operations
- `A @ A`: inner dimensions 3 and 2 do not match.
- `A * B`: shapes `(2,3)` and `(3,2)` cannot broadcast. Starting from the right, dimensions must match or one must be 1; 3 and 2 do not meet that rule.
- `v * w[:2]`: length-3 and length-2 arrays cannot broadcast.
- `v @ 2`: matrix multiplication does not accept a scalar operand. Use `v * 2` for scaling.

Switching operators changes the operation and its shape requirements. It does not guarantee that the new operation is valid or mathematically intended. The catches below let all four error messages be displayed without stopping the notebook.

In [ ]:
invalid_operations = [
    ("A @ A", lambda: A_at @ A_at),
    ("A * B", lambda: A_at * B_at),
    ("v * w[:2]", lambda: v * w[:2]),
    ("v @ 2", lambda: v @ 2)]
operator_error_count = 0
for label, operation in invalid_operations:
    try:
        print(label, "returned:", operation())
    except (ValueError, TypeError) as error:
        operator_error_count += 1
        print(label, "->", type(error).__name__ + ":", str(error))
print("Expected invalid operations caught:", operator_error_count)


### (d) Absolute and relative tolerances
For finite entries, `allclose(a, b)` checks $|a_i-b_i|\le\mathrm{atol}+\mathrm{rtol}|b_i|$ for every pair. The reference is the second argument, `b`. Defaults are `rtol=1e-5` and `atol=1e-8`.

Here the differences are approximately $10^{-6}$ and $5\times10^{-9}$. With defaults, the first allowance is $10^{-8}+10^{-5}$ and the second is $10^{-8}$. Both comparisons pass. The code prints each entry's decision as well as the final Boolean. `np.isclose` applies the same condition but returns one Boolean per entry; `np.allclose` returns a single Boolean for the entire comparison.

In [ ]:
tolerance_a = np.array([1.000001, 0.000000005])
tolerance_b = np.array([1.0, 0.0])
print("Defaults written explicitly:",
      np.allclose(tolerance_a, tolerance_b, rtol=1e-5, atol=1e-8))
print("Differences:", np.abs(tolerance_a - tolerance_b))
tolerance_settings = [(1e-5, 1e-8), (1e-7, 1e-8),
                      (1e-5, 1e-10), (0, 1e-5)]
tolerance_results = []
for rtol, atol in tolerance_settings:
    allowance = atol + rtol * np.abs(tolerance_b)
    entry_results = np.isclose(tolerance_a, tolerance_b, rtol=rtol, atol=atol)
    all_result = np.allclose(tolerance_a, tolerance_b, rtol=rtol, atol=atol)
    tolerance_results.append(bool(all_result))
    print("rtol:", rtol, "atol:", atol)
    print("Allowed differences:", allowance)
    print("Entries pass:", entry_results, "allclose:", all_result)


The results are:

| `rtol` | `atol` | Entry 1 | Entry 2 | `allclose` |
|---|---|---|---|---|
| `1e-5` | `1e-8` | passes | passes | `True` |
| `1e-7` | `1e-8` | fails | passes | `False` |
| `1e-5` | `1e-10` | passes | fails | `False` |
| `0` | `1e-5` | passes | passes | `True` |

When the reference value is zero, `rtol * abs(b_i)` is zero for every choice of `rtol`; only `atol` controls that comparison. Setting `rtol=0` makes the test use a fixed absolute allowance everywhere. A tighter tolerance is not automatically more appropriate: choose tolerances that reflect the numerical accuracy required by the problem.

## Question 2 — Asymptotic notation
### (a) Explicit bounds and limits
For every integer $n\ge1$,
$$|3n^2+5n+7|\le 3n^2+5n^2+7n^2=15n^2.$$
Thus $C=15$ and $n_0=1$ work. Similarly,
$$|4n^3-2n+1|\le4n^3+2n+1\le7n^3,$$
so $C=7$ and $n_0=1$ work for the cubic. These constants need not be the smallest possible.

The relevant ratios are
$$\frac{3n^2+5n+7}{n^2}=3+\frac5n+\frac7{n^2}\longrightarrow3,$$
$$\frac{4n^3-2n+1}{n^3}=4-\frac2{n^2}+\frac1{n^3}\longrightarrow4,$$
$$\frac{4n^3-2n+1}{n^4}=\frac4n-\frac2{n^3}+\frac1{n^4}\longrightarrow0.$$
Therefore the quadratic is **not** $o(n^2)$; the cubic is **not** $o(n^3)$ but **is** $o(n^4)$.

### (b) Sum and product rules
There are constants $C_1,C_2>0$ and a common sufficiently large threshold such that $|f|\le C_1g$ and $|h|\le C_2k$. The triangle inequality gives
$$|f+h|\le C_1g+C_2k\le\max(C_1,C_2)(g+k),$$
which proves $f+h=O(g+k)$. Also
$$|fh|\le C_1C_2gk,$$
so $fh=O(gk)$. All bounds hold beyond the maximum of the original thresholds.

For an $O(n)$ term and an $O(n^2)$ term, their sum is $O(n+n^2)=O(n^2)$ because $n+n^2\le2n^2$ for $n\ge1$. Their product is $O(n^3)$. These are upper bounds, not necessarily exact growth rates; cancellation can make a sum smaller.

### (c) Little-$o$ sums
The first implication is false. Take $g(n)=n$, $f(n)=1$ and $h(n)=n$. Then $f=o(g)$ and $h=O(g)$, but
$$\frac{f+h}{g}=\frac1n+1\longrightarrow1,$$
so $f+h$ is not $o(g)$.

If both $f=o(g)$ and $h=o(g)$, choose any $\varepsilon>0$. Beyond a sufficiently large common threshold, $|f|\le(\varepsilon/2)g$ and $|h|\le(\varepsilon/2)g$. Hence $|f+h|\le\varepsilon g$, which proves $f+h=o(g)$.

### (d) An upper bound need not be tight
No. The function $f(n)=n$ is $O(n^2)$ and also $O(n)$, since $n\le n^2$ for $n\ge1$. Knowing an $O(n^2)$ upper bound does not exclude a smaller upper bound. Big-$O$ alone is not an assertion of an exact growth rate.

## Question 3 — Temperature and reaction rate
### (a) Transforming the model
Starting with $r(T)=C\exp(-B/T)$, take logarithms of the positive numerical rate values:
$$\log r=\log C-B\left(\frac1T\right).$$
Use input $z=1/T$ and response $\ell=\log r$. The intercept is $\alpha=\log C$ and the slope is $\beta=-B$. Thus $C=\exp(\alpha)$ and $B=-\beta$.

Rates must be positive to take their logarithms. Absolute temperatures are positive in this physical model; $1/T$ is undefined at zero. We do not take $\log T$ here.

### (b) Design matrix and fitted parameters
Each design row is $[1,1/T_i]$. Since $T$ is in kelvin, $B$ has units kelvin, so $B/T$ is dimensionless. $C$ has the same rate unit as $r$. Taking the logarithm means using numerical rates in the fixed unit specified in the question.

In [ ]:
T = np.array([280., 290., 300., 310., 320.,
              330., 340., 350., 360., 370.])
r = np.array([0.39, 0.50, 0.66, 0.88, 1.12,
              1.40, 1.70, 2.13, 2.56, 3.08])
z = 1.0 / T
log_r = np.log(r)
A_rate = np.column_stack([np.ones(len(T)), z])
theta_rate, sums_rate, rank_rate, singular_rate = np.linalg.lstsq(
    A_rate, log_r, rcond=None)
alpha, beta = theta_rate
C_fit = np.exp(alpha)
B_fit = -beta

def rate_model(temperature):
    return C_fit * np.exp(-B_fit / temperature)

print("Design matrix shape:", A_rate.shape)
print("Intercept and slope:", theta_rate)
print("C in the chosen rate unit:", C_fit)
print("B in kelvin:", B_fit)
print("Design rank:", rank_rate)


### (c) The line, original curve and an interpolated value
The transformed horizontal coordinate $1/T$ runs in the opposite direction to temperature, so sort that coordinate when drawing the fitted line. The second panel uses kelvin directly. The smooth curve evaluates the fitted function at many temperatures; it does not connect noisy measurements point by point.

In [ ]:
T_grid = np.linspace(280, 370, 200)
z_grid = np.linspace(z.min(), z.max(), 200)
fig, axs = plt.subplots(1, 2, figsize=(11, 4))
axs[0].scatter(z, log_r, label="Measurements")
axs[0].plot(z_grid, alpha + beta * z_grid, color="tab:orange",
            label="Fitted line")
axs[0].set_xlabel("1 / temperature (1/K)")
axs[0].set_ylabel("Log of rate in the chosen unit")
axs[0].set_title("Transformed relationship")
axs[1].scatter(T, r, label="Measurements")
axs[1].plot(T_grid, rate_model(T_grid), color="tab:orange",
            label="Fitted reaction-rate model")
axs[1].set_xlabel("Temperature (K)")
axs[1].set_ylabel("Rate (chosen unit)")
axs[1].set_title("Original relationship")
for ax in axs:
    ax.legend()
    ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()
print("Predicted rate at 335 K:", rate_model(335.0))


The curve closely follows the overall rise in the observations. The prediction at 335 K is **interpolation**, because it lies between measured temperatures 330 K and 340 K, within the observed range.

### (d) Which error was minimized?
The fitted coefficients minimize the **sum of squared log-rate residuals**,
$$\sum_i[\log r_i-(\alpha+\beta/T_i)]^2.$$
This is not the sum of squared errors in the original rates. Taking a square root of an error sum does not change its minimizer, but taking logarithms of the responses changes the fitting criterion.

The original root sum of squared errors has the rate unit. The log residuals are dimensionless: each is $\log(r_i/\hat r_i)$ with matching rate units. Their numerical sizes are not directly comparable. The original root sum is not RMSE; RMSE would divide it by $\sqrt{10}$.

In [ ]:
rate_predictions = rate_model(T)
rate_root_sse = np.linalg.norm(r - rate_predictions)
log_sse = np.sum((log_r - A_rate @ theta_rate) ** 2)
print("Root sum of squared errors in original rate units:", rate_root_sse)
print("Sum of squared log-rate errors:", log_sse)
print("Returned transformed-error sum:", sums_rate)
print("Transformed error agrees:", np.allclose(log_sse, sums_rate[0]))


## Question 4 — Solving without an inverse
### (a) A small exact system
With $x=(u,v,w)^T$, the matrix rows contain the coefficients in that order. The solution is $(1.25,2.5,1.25)^T$. We check the equations rather than compare floating-point values with exact equality.

In [ ]:
A_small = np.array([[2., 1., 0.],
                    [1., 3., 1.],
                    [0., 1., 2.]])
b_small = np.array([5., 10., 5.])
x_small_solve = np.linalg.solve(A_small, b_small)
x_small_inverse = np.linalg.inv(A_small) @ b_small
print("solve:", x_small_solve)
print("inverse then multiply:", x_small_inverse)
print("Substituted left-hand sides:", A_small @ x_small_solve)
print("Residual:", b_small - A_small @ x_small_solve)
print("Equations satisfied:", np.allclose(A_small @ x_small_solve, b_small))
print("Methods agree:", np.allclose(x_small_solve, x_small_inverse))


### (b) Timing on a larger system
For any nonzero vector $v$, $v^TAv=\|Mv\|^2+\|v\|^2>0$ when $A=M^TM+I$. Thus $A$ is positive definite and nonsingular in exact arithmetic.

The timer measures each complete solve, including construction of the inverse in the second method. `lambda: ...` supplies a no-argument function that the timer calls later. Each batch contains three calls; the minimum of three batch times divided by three gives seconds per solution for the fastest batch. Data creation and printing are outside the timed call.

In [ ]:
rng = np.random.default_rng(587)  # Fixed seed
n_large = 300
M = rng.normal(size=(n_large, n_large))
A_large = M.T @ M + np.eye(n_large)
x_ref_large = np.ones(n_large)
b_large = A_large @ x_ref_large

solve_times = timeit.repeat(lambda: np.linalg.solve(A_large, b_large),
                            number=3, repeat=3)
inverse_times = timeit.repeat(lambda: np.linalg.inv(A_large) @ b_large,
                              number=3, repeat=3)
solve_seconds = min(solve_times) / 3
inverse_seconds = min(inverse_times) / 3
print("solve seconds per solution:", solve_seconds)
print("inverse route seconds per solution:", inverse_seconds)
print("inverse time / solve time:", inverse_seconds / solve_seconds)
print("Faster in this run:", "solve" if solve_seconds < inverse_seconds
      else "inverse then multiply")
print("Entries in a solution vector:", n_large)
print("Entries in the inverse:", n_large * n_large)


Both standard dense routes have $O(n^3)$ leading work, but different constants and operations. Forming an inverse produces 90,000 entries here, while the requested answer has 300. `solve` also needs matrix-factor storage; this comparison does not imply its total workspace is only 300 numbers. Timing order can vary, especially for small examples and different BLAS implementations.

### (c) Accuracy on Hilbert matrices
A small residual says the computed vector almost satisfies the supplied equations. A small **solution error** says it is close to the reference vector. They need not go together for an ill-conditioned system.

The code below prints both relative quantities for each method. The second measures error against the intended all-ones reference; the stored $b$ is already rounded when calculated. The all-ones vector is the intended reference, not a high-precision solution of the rounded system.

In [ ]:
hilbert_results = []
for size in [8, 10, 12]:
    i = np.arange(size)
    H = 1.0 / (i[:, None] + i[None, :] + 1)
    x_ref = np.ones(size)
    b_H = H @ x_ref
    print("Size:", size)
    for method in ["solve", "inverse"]:
        try:
            if method == "solve":
                x_H = np.linalg.solve(H, b_H)
            else:
                x_H = np.linalg.inv(H) @ b_H
            relative_residual = np.linalg.norm(b_H - H @ x_H) / np.linalg.norm(b_H)
            relative_error = np.linalg.norm(x_H - x_ref) / np.linalg.norm(x_ref)
            hilbert_results.append((size, method, relative_residual, relative_error))
            print(method, "relative residual:", relative_residual,
                  "relative reference error:", relative_error)
        except np.linalg.LinAlgError as error:
            print(method, "failed:", error)


The saved run shows the inverse route leaving larger residuals, and the all-ones solution becoming harder to recover as the Hilbert matrix grows. `solve` can have a residual near machine precision while its reference error is noticeable. Small residuals therefore do not guarantee small solution errors. Results depend on rounding and the numerical library; neither method removes the sensitivity of the problem.

### (d) Practical choice
Use `np.linalg.solve(A, b)` when $A$ is square and nonsingular and you only need $x$. It avoids explicitly forming an inverse, generally does less work for one right-hand side, and avoids an additional inverse-multiplication error route. Agreement on a small well-behaved system does not establish that inversion is equally suitable for larger or sensitive systems. `solve` is not a cure for ill-conditioning; the Hilbert experiment demonstrates that distinction.

## Question 5 — QR and least squares
### (a) Independent columns
For this $6\times3$ design, reduced QR gives $Q$ of shape $(6,3)$ and upper-triangular $R$ of shape $(3,3)$. Its columns are independent, so $R$ is nonsingular. Orthogonality of the least-squares residual gives
$$Q^T(y-QR\theta)=0\quad\Rightarrow\quad R\theta=Q^Ty.$$
We factor $A$ itself, not $A^TA$.

In [ ]:
A_qr = np.array([[1., 0., 0.],
                 [1., 1., 1.],
                 [1., 2., 0.],
                 [1., 3., 1.],
                 [1., 4., 0.],
                 [1., 5., 1.]])
y_qr = np.array([1., 2., 2., 4., 4., 5.])

def qr_fit(A, y):
    Q, R = np.linalg.qr(A, mode="reduced")
    return solve_triangular(R, Q.T @ y)

Q, R = np.linalg.qr(A_qr, mode="reduced")
theta_qr = qr_fit(A_qr, y_qr)
theta_ls = np.linalg.lstsq(A_qr, y_qr, rcond=None)[0]
print("Rank:", np.linalg.matrix_rank(A_qr))
print("Q and R shapes:", Q.shape, R.shape)
print("QR coefficients:", theta_qr)
print("lstsq coefficients:", theta_ls)
print("Coefficients agree:", np.allclose(theta_qr, theta_ls))
print("QR residual norm:", np.linalg.norm(y_qr - A_qr @ theta_qr))
print("lstsq residual norm:", np.linalg.norm(y_qr - A_qr @ theta_ls))


### (b) Timing both full calculations
Use the same fixed arrays. Each QR call recomputes the factorisation, projection and triangular solve. The `lstsq` call also performs its full computation. Each fastest batch is divided by 100, the calls per batch.

In [ ]:
qr_times = timeit.repeat(lambda: qr_fit(A_qr, y_qr),
                         number=100, repeat=3)
ls_times = timeit.repeat(
    lambda: np.linalg.lstsq(A_qr, y_qr, rcond=None),
    number=100, repeat=3)
qr_seconds = min(qr_times) / 100
ls_seconds = min(ls_times) / 100
print("QR seconds per fit:", qr_seconds)
print("lstsq seconds per fit:", ls_seconds)
print("Faster for this matrix in this run:",
      "QR" if qr_seconds < ls_seconds else "lstsq")


The printed result answers the question for this experiment. The matrix is small: function-call overhead and library implementation details can matter as much as the arithmetic. This is not evidence of a universal speed ordering.

### (c) Dependent columns
Replacing column 2 by twice column 1 leaves only two independent columns. The fitted values depend on the intercept and the combination $\theta_1+2\theta_2$, using zero-based coefficient indices. A change proportional to $(0,-2,1)^T$ leaves predictions unchanged.

QR still exists, but $R$ is singular in exact arithmetic. A diagonal entry may be merely tiny in floating-point arithmetic. A triangular solver may return very large, cancelling coefficients instead of reporting failure.

In [ ]:
A_dep = A_qr.copy()
A_dep[:, 2] = 2 * A_dep[:, 1]
Q_dep, R_dep = np.linalg.qr(A_dep, mode="reduced")
theta_dep, sums_dep, rank_dep, singular_dep = np.linalg.lstsq(
    A_dep, y_qr, rcond=None)
print("Numerical rank:", rank_dep)
print("R diagonal:", np.diag(R_dep))
print("lstsq coefficients:", theta_dep)
print("lstsq coefficient norm:", np.linalg.norm(theta_dep))
print("lstsq residual norm:", np.linalg.norm(y_qr - A_dep @ theta_dep))
try:
    theta_dep_qr = qr_fit(A_dep, y_qr)
    print("QR coefficients:", theta_dep_qr)
    print("QR coefficient norm:", np.linalg.norm(theta_dep_qr))
    print("QR residual norm:", np.linalg.norm(y_qr - A_dep @ theta_dep_qr))
except np.linalg.LinAlgError as error:
    print("QR triangular solve failed:", error)

null_direction = np.array([0., -2., 1.])
alternative = theta_dep + null_direction
print("Null direction gives zero predictions:",
      np.allclose(A_dep @ null_direction, np.zeros(len(y_qr))))
print("Alternative coefficients:", alternative)
print("Same fitted values:", np.allclose(A_dep @ alternative, A_dep @ theta_dep))
print("Alternative coefficient norm:", np.linalg.norm(alternative))


The minimum-norm result is approximately $(1,0.16,0.32)^T$: the slope combination is $0.16+2(0.32)=0.8$. Among all pairs with this combination, this one has the smallest sum of coefficient squares. Every vector obtained by adding a multiple of $(0,-2,1)^T$ gives the same fitted values in exact arithmetic.

A small residual cannot identify a unique coefficient vector. The simple full-column-rank QR recipe does not make a reliable rank decision or choose a minimum-norm solution. `lstsq` detects numerical rank and selects the minimizing vector with the smallest Euclidean norm. Its returned error-sums array is empty here because the matrix lacks full column rank; calculate the actual residual explicitly.

## Question 6 — Polynomial models on housing data
### (a) CSV to NumPy binary
The header names the columns `elapsed_year` and `price_index`; it is text rather than a numeric measurement, so skip it while loading. `housing.npy` below is created by this cell, not an extra input file.

In [ ]:
csv_file = Path("housing.csv")
if not csv_file.is_file():
    raise FileNotFoundError("Keep housing.csv beside this notebook and run from that folder.")
with csv_file.open() as f:
    for line in list(f)[:4]:
        print(line.rstrip())

housing_from_csv = np.loadtxt("housing.csv", delimiter=",", skiprows=1)
print("Shape:", housing_from_csv.shape)
print("First three numerical rows:")
print(housing_from_csv[:3])
np.save("housing.npy", housing_from_csv)
housing_reloaded = np.load("housing.npy", allow_pickle=False)
print("Round-trip values identical:", np.array_equal(housing_from_csv, housing_reloaded))


### (b) A first look and input scaling
`t` is elapsed time from the first observation, not a calendar year. The price index is a single index series; each row is not an individual house. Divide `t` by 16 for polynomial calculations. This changes the basis scale but not the polynomial family: each power of $t/16$ is a constant multiple of the corresponding power of $t$. Keep the plotted horizontal axis in years.

In [ ]:
housing = np.load("housing.npy", allow_pickle=False)
t_housing = housing[:, 0]
y_housing = housing[:, 1]
x_housing = t_housing / 16.0
fig, ax = plt.subplots()
ax.scatter(t_housing, y_housing, color="tab:blue")
ax.set_xlabel("Years since first observation")
ax.set_ylabel("Housing price index")
ax.set_title("Housing observations")
ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()


The index rises, then falls, then rises again. A single straight line cannot represent all of those bends. That observation suggests exploring additional features; it does not by itself select a polynomial degree.

### (c) One reproducible split
The training observations determine coefficients; validation observations help choose the degree; test observations are reserved for the final evaluation. Use one split for all candidates so that changing the split does not confound the comparison. Seed 587 is arbitrary and fixed for reproducibility.

In [ ]:
rng = np.random.default_rng(587)  # Fixed seed: reproducible split
indices = rng.permutation(len(y_housing))
train = indices[:21]
validation = indices[21:27]
test = indices[27:]
print("Training, validation and test counts:", len(train), len(validation), len(test))
print("Training row indices:", train)
print("Validation row indices:", validation)
print("Test row indices:", test)


### (d) Building polynomial columns and comparing errors
For degree 3, the columns are $1,x,x^2,x^3$. The function below builds exactly that pattern with a loop and `np.column_stack`. Coefficients are in increasing power order, so the same function must construct both fitting and prediction matrices. RMSE is in price-index units. `round(value, 4)` only shortens printed output; fitting uses the unrounded numbers.

In [ ]:
def design_matrix(x, degree):
    columns = [np.ones(len(x))]
    for power in range(1, degree + 1):
        columns.append(x ** power)
    return np.column_stack(columns)

def rmse(observed, predicted):
    return np.sqrt(np.mean((observed - predicted) ** 2))

degrees = [1, 2, 3, 5, 8]
polynomial_fits = {}
polynomial_errors = []
for degree in degrees:
    A_train = design_matrix(x_housing[train], degree)
    coefficients, sums, rank, singular = np.linalg.lstsq(
        A_train, y_housing[train], rcond=None)
    train_prediction = A_train @ coefficients
    validation_prediction = design_matrix(x_housing[validation], degree) @ coefficients
    train_rmse = rmse(y_housing[train], train_prediction)
    validation_rmse = rmse(y_housing[validation], validation_prediction)
    polynomial_fits[degree] = coefficients
    polynomial_errors.append((degree, train_rmse, validation_rmse, rank))

print("Degree | Training RMSE | Validation RMSE | Design rank")
for degree, train_error, validation_error, rank in polynomial_errors:
    print(degree, round(train_error, 4), round(validation_error, 4), rank)


### (e) Comparing curves and error plots
The grid is sorted even though the training row indices are shuffled. Evaluate on its scaled values, but plot against elapsed years. Both colours below are available for model development; test responses are not plotted yet. Shared axes prevent a misleading change in apparent fit quality between panels.

In [ ]:
t_grid = np.linspace(t_housing.min(), t_housing.max(), 400)
x_grid = t_grid / 16.0
fig, axs = plt.subplots(2, 3, figsize=(13, 7), sharex=True, sharey=True)
for ax, degree in zip(axs.flat, degrees):
    curve = design_matrix(x_grid, degree) @ polynomial_fits[degree]
    ax.plot(t_grid, curve, color="tab:green", label="Polynomial fit")
    ax.scatter(t_housing[train], y_housing[train], label="Training", s=24)
    ax.scatter(t_housing[validation], y_housing[validation],
               label="Validation", marker="s", s=32)
    ax.set_title("Degree " + str(degree))
    ax.set_xlabel("Elapsed years")
    ax.set_ylabel("Housing price index")
    ax.grid(alpha=0.2)
axs.flat[-1].set_visible(False)
axs.flat[0].legend(fontsize=8)
fig.tight_layout()
plt.show()

train_errors = np.array([row[1] for row in polynomial_errors])
validation_errors = np.array([row[2] for row in polynomial_errors])
fig, ax = plt.subplots()
ax.plot(degrees, train_errors, "o-", label="Training")
ax.plot(degrees, validation_errors, "s-", label="Validation")
ax.set_xticks(degrees)
ax.set_xlabel("Polynomial degree")
ax.set_ylabel("RMSE (price-index units)")
ax.set_title("Errors before examining the test set")
ax.legend()
ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()


Degrees 1 and 2 miss much of the rise–fall–rise structure. Degrees 3 and 5 capture more of it. Degree 8 follows finer variations; extra bends deserve inspection, but they are not by themselves proof that its validation performance must be worse.

Training RMSE decreases for these nested polynomial families, apart from numerical rounding. Validation RMSE need not decrease when more powers are added. Here the validation scores of degrees 5 and 8 are fairly close; a preference for a simpler degree 5 could be justified if decided before the test results. For one reproducible worked solution we use the **lowest validation RMSE** rule and do not revise it after testing.

### (f) Select, refit and evaluate once
Choose the degree from validation scores only. Refit that chosen family with 27 training-plus-validation observations, then evaluate the six test observations once. The constant baseline uses the mean of those same 27 fitting observations. The final plot may now show the test points because selection is finished.

In [ ]:
chosen_degree = degrees[int(np.argmin(validation_errors))]
print("Degree selected from validation RMSE only:", chosen_degree)
fit_indices = np.concatenate([train, validation])
A_refit = design_matrix(x_housing[fit_indices], chosen_degree)
final_coefficients = np.linalg.lstsq(A_refit, y_housing[fit_indices], rcond=None)[0]
final_test_predictions = design_matrix(x_housing[test], chosen_degree) @ final_coefficients
baseline_value = np.mean(y_housing[fit_indices])
baseline_predictions = np.full(len(test), baseline_value)
test_rmse = rmse(y_housing[test], final_test_predictions)
baseline_test_rmse = rmse(y_housing[test], baseline_predictions)
print("Selected-polynomial test RMSE:", test_rmse)
print("Constant-baseline test RMSE:", baseline_test_rmse)
print("Polynomial beats this baseline on this split:", test_rmse < baseline_test_rmse)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(t_grid, design_matrix(x_grid, chosen_degree) @ final_coefficients,
        color="tab:green", label="Refitted degree " + str(chosen_degree))
ax.axhline(baseline_value, color="gray", linestyle="--", label="Constant baseline")
ax.scatter(t_housing[train], y_housing[train], label="Training", s=28)
ax.scatter(t_housing[validation], y_housing[validation], label="Validation", marker="s")
ax.scatter(t_housing[test], y_housing[test], label="Test", color="tab:red", marker="*", s=100)
ax.set_xlabel("Years since first observation")
ax.set_ylabel("Housing price index")
ax.set_title("Final model: selection completed before testing")
ax.legend(fontsize=9)
ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()


This test comparison describes one small shuffled split of the supplied housing-index observations. It is not a claim about forecasting future prices or independent repeated housing-market experiments. The fitted family and degree remain fixed after the test is examined. A degree chosen using test error would no longer have an untouched test evaluation.

## Files and rerunning
The only required input is `housing.csv`. Question 6 creates `housing.npy`; rerunning replaces that local derived file. The notebook contains the functions used for fitting and error calculations, so no separate solution module is needed. The Markdown export includes the saved outputs and links to its companion image folder; keep both together when sharing or converting to HTML.